In [ ]:
import torch

from dataset_loaders import build_data_loaders
from models.autoencoder import AbstractAutoencoder
from utils.checkpoints import load_ae_from_path, load_from_wandb
from utils.config import DatasetConfig
from utils.visualisation import plot_latent_space, show, show_comparison

In [ ]:
ae_path = load_from_wandb("autoencoder_mnist")
ae = load_ae_from_path(ae_path, device=torch.device("mps"))

In [ ]:
dataset_cfg = DatasetConfig(
    name="mnist",
    channels=3,
    height=28,
    width=28,
    num_classes=10,
)

dataloader, test_loader = build_data_loaders(dataset_cfg, batch_size=64, num_workers=0)

In [ ]:
images, labels = next(iter(test_loader))

with torch.no_grad():
    recon, mu, log_var, z = ae(images)
    recon = torch.sigmoid(recon)

show_comparison(images, recon)


In [ ]:
def ae_latent_space(model: AbstractAutoencoder, data_loader: torch.utils.data.DataLoader,
                    title="Latent Space of MNIST Autoencoder"):
    samples = []
    sampled_labels = []
    for image, label in data_loader:
        latent = model.encode(image)
        samples.append(latent)
        sampled_labels.append(label)

    samples = torch.cat(samples, dim=0)
    sampled_labels = torch.cat(sampled_labels, dim=0)
    plot_latent_space(samples, sampled_labels, title=title)

In [ ]:
#versions = ["v29", "v30"]
versions = ["latest"]
for version in versions:
    ae_tmp_path = load_from_wandb("autoencoder_mnist", tag=version)
    ae_tmp = load_ae_from_path(ae_tmp_path, device=torch.device("mps"))
    with torch.no_grad():
        ae_latent_space(ae_tmp, test_loader, title=f"Latent Space of MNIST Autoencoder {version}")